In [2]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** 12
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line):

Daniel Lei
Turki 
Ruoxi
Grace

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

here is a test - hello there why is it not working oooooooooo

pls work

final git push

banana



wowoowowowowowo

In [125]:

tickers_df = pd.read_csv('Tickers_Example.csv', header = None, dtype=str)

start_date = '2025-10-01'
end_date = '2026-10-01'

ticker_list = []
for ticker in tickers_df.iloc[:, 0].dropna():
    ticker = ticker.replace(' ', '').upper()
    # skip blanks any title rows and duplicates (e.g. RY.TO listed twice)
    if ticker == '' or ticker == 'TICKER' or ticker in ticker_list:
        continue

    ticker_list.append(ticker)

In [126]:
symbols = ticker_list + ['^GSPC', 'CAD=X']

close = pd.DataFrame()
volume = pd.DataFrame()
for attempt in range(3):
    missing = [t for t in symbols if t not in close.columns]
    if len(missing) == 0:
        break
    data = yf.download(missing, start=start_date, end=end_date, auto_adjust=False, progress=False)
    if data.empty:
        continue
    new_close = data['Close'].dropna(axis=1, how='all')
    close = pd.concat([close, new_close], axis=1)
    volume = pd.concat([volume, data['Volume'][new_close.columns]], axis=1)

print('No price data (invalid or delisted):', [t for t in ticker_list if t not in close.columns])

$BK: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
$AGN: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found
$CELG: possibly delisted; no timezone found
$MON: possibly delisted; no timezone found

5 Failed downloads:
['BK']: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
['AGN', 'RTN', 'CELG', 'MON']: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found
$MON: possibly delisted; no timezone found
$CELG: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)

5 Failed downloads:
['AGN', 'RTN', 'MON', 'CELG']: possibly delisted; no timezone found
['BK']: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
$AGN: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found
$MON: possibly delisted; no timezone found
$CELG: possibly delisted; no timezone fou

No price data (invalid or delisted): ['AGN', 'BK', 'CELG', 'MON', 'RTN']


In [127]:
stock_volume = volume.drop(columns=['^GSPC', 'CAD=X'], errors='ignore')

# months with fewer than 18 trading days are ignored
days_per_month = stock_volume.groupby(stock_volume.index.to_period('M')).transform('count')
stock_volume = stock_volume.where(days_per_month >= 18)

mean_volumes = stock_volume.mean()
filtered_tickers = mean_volumes[mean_volumes >= 5000].index.tolist()
filtered_tickers

['AAPL',
 'ABBV',
 'ABT',
 'ACN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [128]:
allowed_exchanges = ['NMS', 'NGM', 'NCM', 'NYQ', 'NYS', 'ASE', 'PCX', 'BTS', 'BATS', 'NAS', 'TOR', 'VAN', 'CNQ', 'NEO']

info_stocks = {}
for ticker in filtered_tickers:
    info = yf.Ticker(ticker).get_info()
    if info.get('quoteType') != 'EQUITY':
        print(f'Removed (not a stock): {ticker}')
    elif info.get('currency') not in ['CAD', 'USD']:
        print(f'Removed (currency is {info.get("currency")}): {ticker}')
    elif info.get('exchange') not in allowed_exchanges:
        print(f'Removed (not listed in Canada/US, exchange is {info.get("exchange")}): {ticker}')
    else:
        info_stocks[ticker] = {'Name': info.get('shortName', ticker),
                               'Sector': info.get('sector') or 'Unknown',
                               'Currency': info['currency']}

info_stocks = pd.DataFrame(info_stocks).T
filtered_tickers = info_stocks.index.tolist()
filtered_tickers

['AAPL',
 'ABBV',
 'ABT',
 'ACN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [129]:
seen = {}
for t in filtered_tickers.copy():
    key = info_stocks.loc[t, 'Name'].lower().replace('.', '').replace(',', '').strip()
    if key not in seen:
        seen[key] = t
    elif mean_volumes[t] > mean_volumes[seen[key]]:
        print(f'Removed (second listing of the same company): {seen[key]}')
        filtered_tickers.remove(seen[key])
        seen[key] = t
    else:
        print(f'Removed (second listing of the same company): {t}')
        filtered_tickers.remove(t)

info_stocks = info_stocks.loc[filtered_tickers]
filtered_tickers

['AAPL',
 'ABBV',
 'ABT',
 'ACN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [130]:
temp_list = ticker_data['Close']
ticker_list = temp_list.columns.tolist()
ticker_list


NameError: name 'ticker_data' is not defined

In [ ]:
fx = close['CAD=X'].ffill()
market = close['^GSPC'].ffill() * fx

prices = close[filtered_tickers].ffill()
for t in filtered_tickers:
    if info_stocks.loc[t, 'Currency'] == 'USD':
        prices[t] = prices[t] * fx

returns = prices.pct_change().iloc[1:]
market_returns = market.pct_change().iloc[1:]

beta = {}
for t in filtered_tickers:
    beta[t] = returns[t].cov(market_returns) / market_returns.var()
beta = pd.Series(beta)

standard_deviation = returns.std()
average_return = returns.mean()

pd.DataFrame({'Beta': beta, 'Daily std': standard_deviation, 'Avg daily return': average_return})

,Beta,Daily std,Avg daily return
AAPL,0.736774,0.015533,0.001223
ABBV,0.052005,0.016383,0.000471
ABT,0.132084,0.016980,-0.000947
ACN,0.337659,0.028184,-0.000625
AIG,0.102999,0.015198,-0.000074
AMZN,1.367508,0.021372,0.000768
BA,1.293161,0.021654,-0.000258
BAC,0.761664,0.014034,0.000449
BB.TO,1.406293,0.037817,0.003238
BIIB,0.483250,0.021318,0.001780


In [ ]:
def apply_caps(raw, sectors, max_stock=0.15, max_sector=0.40):
    w = raw / raw.sum()
    frozen = pd.Series(False, index=w.index)
    for i in range(100):
        over = w > max_stock + 1e-12
        w[over] = max_stock
        frozen = frozen | over
        for sector in sectors.unique():
            members = sectors[sectors == sector].index
            total = w[members].sum()
            if total > max_sector + 1e-12:
                w[members] = w[members] * max_sector / total
                frozen[members] = True
        extra = 1 - w.sum()
        if extra < 1e-12:
            break
        free = ~frozen
        w[free] = w[free] + extra * raw[free] / raw[free].sum()
    return w

In [ ]:
buy = yf.download(filtered_tickers + ['CAD=X'], start='2026-11-16', end='2026-11-28', auto_adjust=False, progress=False)['Close'].ffill()
buy_prices = buy.iloc[-1]
fx_buy = buy_prices['CAD=X']
print('Purchase prices are from', buy.index[-1].date())

def build_portfolio(weights, budget):
    price_cad = buy_prices[weights.index].copy()
    for t in weights.index:
        if info_stocks.loc[t, 'Currency'] == 'USD':
            price_cad[t] = price_cad[t] * fx_buy

    low = 0
    high = budget
    for i in range(100):
        invest = (low + high) / 2
        shares = weights * invest / price_cad
        fees = (shares * 0.001).clip(upper=2.15).sum() * fx_buy
        if invest + fees <= budget - 0.01:
            low = invest
        else:
            high = invest

    shares = weights * low / price_cad
    fees = (shares * 0.001).clip(upper=2.15).sum() * fx_buy
    value = shares * price_cad

    table = pd.DataFrame({'Ticker': weights.index,
                          'Price': buy_prices[weights.index].values,
                          'Currency': info_stocks.loc[weights.index, 'Currency'].values,
                          'Shares': shares.values,
                          'Value': value.values,
                          'Weight': (value / value.sum()).values})
    table.index = range(1, len(table) + 1)
    return table, fees

def history_stats(weights):
    units = weights / prices[weights.index].iloc[0]
    value = (prices[weights.index] * units).sum(axis=1)
    daily = value.pct_change().dropna()
    return {'Total return': value.iloc[-1] / value.iloc[0] - 1,
            'Daily std': daily.std(),
            'Annualized std': daily.std() * 252 ** 0.5}

def comparison(weights):
    equal = pd.Series(1 / len(weights), index=weights.index)
    return pd.DataFrame({'Chosen weights': history_stats(weights), 'Equal weights': history_stats(equal)}).T


Purchase prices are from 2026-11-20


After removing all the unwanted stocks, get data for each of the remaining stocks that will help us determine if they are stable and optimal for us to add to our portfolio:
Standard deviation
Tells us how volatile/risky the stock is, and we will look for stocks that have lower standard deviations (close to 0) to reduce the volatility and risk of our portfolio
Beta to the S&P 500
Tells us how closely the stock moves with the market. We want to look for stocks that have lower betas (close to 0 in absolute value) because it will mean that the stock's movements aren't very correlated with the market, and don't particularly go up or down when the market moves
Expected Monthly Returns
Tells us what percentage return we should expect to see per month. We want to look for stocks that have lower expected monthly returns (close to 0 in absolute value) because we want a portfolio that gives as close to 0% returns as possible

| Signal | Why it points toward higher return | Weight |
|---|---|---|
| **Beta** (Chapter 15) | In a rising market, a beta-1.5 stock tends to move about 1.5× the market. CAPM: more market risk earns more expected return. | 40% |
| **12-1 month momentum** | Past-year winners (skipping the latest month, which tends to bounce back) have tended to keep winning (Jegadeesh & Titman, 1993). Research evidence, not from our slides. | 40% |
| **Revenue growth** (Yahoo) | Our client wants *growth*; fast-growing sales support a company's story. Yahoo's data is patchy, so it gets less weight and **a missing value counts as neutral**. | 20% |

**Selection:** top 12 by score, at most 4 per sector. **Weighting:** proportional to score (shifted so the lowest of the 12 is still positive),
then the 15% / 40% caps are applied and any excess is spread over the uncapped stocks.
**What we deliberately did *not* use:** dividends — the competition ignores dividends, so they earn nothing in this contest.

In [ ]:
raw_a = score_a[chosen_a] - score_a[chosen_a].min() + 1
weights_a = apply_caps(raw_a, info_stocks.loc[chosen_a, 'Sector'])
Portfolio_Final_A, fees_a = build_portfolio(weights_a, 1000000)
display(Portfolio_Final_A)

,Ticker,Price,Currency,Shares,Value,Weight
1,CAT,510.450012,USD,204.934704,149998.733419,0.150000
2,BB.TO,11.100000,CAD,12772.020735,141769.435029,0.141771
3,TXN,263.750000,USD,306.525863,115925.360877,0.115926
4,QCOM,149.940002,USD,449.638517,96671.817698,0.096673
5,C,110.620003,USD,603.914773,95791.766584,0.095793
6,MRK,133.720001,USD,419.194364,80376.792506,0.080377
7,TD.TO,162.050003,CAD,431.846658,69980.752284,0.069981
8,AMZN,246.820007,USD,177.998257,62996.290221,0.062997
9,BIIB,249.589996,USD,136.179354,48736.833622,0.048737
10,RY.TO,263.049988,CAD,176.967541,46551.309610,0.046552


### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.
Client A aims for the maximum return and is willing to take high risks. As expected returns cannot be observed, I have measured each stock against three factors and aggregated them using their z-scores (their distance from the mean of the entire set). Beta will account for 40%: stock with beta greater than 1 will move even more upwards than the movement of the market, and hence, the theory suggests that investors get higher compensation for market risk according to CAPM. Momentum over 12-1 month period will get 40%. 20% for revenue growth that corresponds to the growth strategy, although it is given less weight since the data for Yahoo is unreliable and the absence of a figure means a neutral score. I have purchased the top 12 based on their scores, where each sector is allowed a maximum of 4 stocks so that no single sector will be dominant. This was done intentionally since the contest gives recognition solely based on the best performing stock over five days, and with fewer stocks, there is greater variation in performance. Higher scores receive larger allocations with the stock limit of 15% and the sector limit of 40%. 

Client A is likely high-scoring (higher beta, growth, strong past year): so stocks like 
AMZN, SHOP.TO, QCOM, LLY, PYPL, AXP, CAT, BA, plus possibly C, BAC, TXN and ACN. BB.TO is very volatile and could score high or low depending on its momentum. The 4-per-sector limit will push the list to spread across tech, financials, industrials and healthcare.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.
If the market falls between Nov 30 and Dec 4, high-beta stocks will fall more than the market, so the beta tilt works against us. Momentum can reverse suddenly, and over only five days luck matters more than any signal. Up to 40% of the portfolio can sit in one sector, so a sector-specific shock would hurt. US stocks also lose value in CAD if the Canadian dollar strengthens.

In [ ]:
past_return = prices.iloc[-21] / prices.iloc[0] - 1

def z(s):
    return (s - s.mean()) / s.std()

score_a = z(beta) + z(past_return)

chosen_a = []
sector_count = {}
for t in score_a.sort_values(ascending=False).index:
    sector = info_stocks.loc[t, 'Sector']
    if sector_count.get(sector, 0) >= 4:
        continue
    chosen_a.append(t)
    sector_count[sector] = sector_count.get(sector, 0) + 1
    if len(chosen_a) == 12:
        break
chosen_a

['CAT',
 'BB.TO',
 'TXN',
 'QCOM',
 'C',
 'MRK',
 'TD.TO',
 'AMZN',
 'BIIB',
 'RY.TO',
 'SHOP.TO',
 'USB']

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.



| Signal | Why it helps reduce risk | Weight |
|---|---|---|
| **Daily STD** | Lower daily volatility means smaller historical price fluctuations and lower individual stock risk. | 40% |
| **Dividend Yield** | Higher dividend yields may indicate established companies that provide regular income. | 25% |
| **Correlation** | Lower correlation between stocks can help reduce overall portfolio volatility through diversification. | 20% |
| **ROA** | Higher Return on Assets indicates that a company uses its assets efficiently to generate profits. | 10% |
| **Profit Margin** | Higher profit margins indicate stronger profitability and may help companies withstand financial difficulties. | 5% |
| **Total** | | **100%** |




### Client B — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

In [201]:
div = []

for i in filtered_tickers:
    ticker = yf.Ticker(i)
    info = ticker.info
    div.append([i, info.get('dividendYield')])
div_df = pd.DataFrame(div, columns=['Ticker', 'Dividend Yield'])

roa = []

for i in filtered_tickers:
    ticker = yf.Ticker(i)
    info = ticker.info
    roa.append([i, info.get('returnOnAssets')])
roa_df = pd.DataFrame(roa, columns=['Ticker', 'Return on Assets'])

pro = []

for i in filtered_tickers:
    ticker = yf.Ticker(i)
    info = ticker.info
    pro.append([i, info.get('profitMargins')])
pro_df = pd.DataFrame(pro, columns=['Ticker', 'Profit Margin'])

ranks = []

for x in roa_df['Return on Assets']:
    r_roa = 1

    for y in roa_df['Return on Assets']:
        if x < y:
            r_roa = r_roa + 1
    ranks.append(r_roa)

roa_df['ROA Rank'] = ranks


ranks = []

for x in pro_df['Profit Margin']:
    r_pro = 1

    for y in pro_df['Profit Margin']:
        if x < y:
            r_pro = r_pro + 1
    ranks.append(r_pro)

pro_df['Profit Margin Rank'] = ranks


ranks = []

for x in div_df['Dividend Yield']:
    r_div = 1

    for y in div_df['Dividend Yield']:
        if x < y:
            r_div = r_div + 1
    ranks.append(r_div)

div_df['Dividend Yield Rank'] = ranks

ranks = []

for x in standard_deviation:
    r_std = 1

    for y in standard_deviation:
        if x > y:
            r_std = r_std + 1
    ranks.append(r_std)

std_df = pd.DataFrame({'Ticker': standard_deviation.index, 'Daily Std': standard_deviation.values, 'Std Rank': ranks})


In [198]:

rankings = pd.merge(div_df, roa_df, on='Ticker')
rankings = pd.merge(rankings, pro_df, on='Ticker')
rankings = pd.merge(rankings, std_df, on='Ticker')
rankings['STD score'] = rankings['Std Rank'] * 0.40
rankings['div score'] = rankings['Dividend Yield Rank'] * 0.25
rankings['PRO score'] = rankings['Profit Margin Rank'] * 0.10
rankings['ROA score'] = rankings['ROA Rank'] * 0.05

rankings['Total Score'] = rankings['STD score'] + rankings['div score'] + rankings['PRO score'] + rankings['ROA score']
rankings = rankings.sort_values(by='Total Score', ascending=True)


rankings



,Ticker,Dividend Yield,Dividend Yield Rank,Return on Assets,ROA Rank,Profit Margin,Profit Margin Rank,Daily Std,Std Rank,STD score,div score,PRO score,ROA score,Total Score
26,RY.TO,2.68,13,0.00963,31,0.33871,3,0.009415,1,0.4,3.25,0.3,1.55,5.50
29,TD.TO,2.76,10,0.00781,33,0.26617,11,0.01035,2,0.8,2.50,1.1,1.65,6.05
22,PFE,6.28,1,0.05593,20,0.06339,30,0.01372,5,2.0,0.25,3.0,1.00,6.25
21,PEP,4.21,6,0.09017,16,0.11091,25,0.013531,4,1.6,1.50,2.5,0.80,6.40
19,MO,5.93,3,0.22733,2,0.39333,1,0.016756,14,5.6,0.75,0.1,0.10,6.55
34,USB,3.85,7,0.01200,30,0.30011,6,0.014262,8,3.2,1.75,0.6,1.50,7.05
8,BAC,2.72,12,0.00944,32,0.28064,9,0.014034,6,2.4,3.00,0.9,1.60,7.90
16,KO,2.19,22,0.09594,13,0.28654,8,0.01227,3,1.2,5.50,0.8,0.65,8.15
28,T.TO,4.73,4,0.03422,28,-0.10313,35,0.014593,10,4.0,1.00,3.5,1.40,9.90
12,BMY,4.29,5,0.10902,9,0.18474,15,0.017122,17,6.8,1.25,1.5,0.45,10.00


In [ ]:
growth = None

if previous != 0:
    growth = (now - previous) / abs(previous)
else: 
    print('there is no')

print(growth)

nan


In [154]:

print(type(standard_deviation))
print(standard_deviation)


<class 'pandas.core.series.Series'>
Ticker
AAPL                                                                0.015533
ABBV                                                                0.016383
ABT                                                                  0.01698
ACN                                                                 0.028184
AIG                                                                 0.015198
AMZN                                                                0.021372
AXP                                                                 0.016921
BA                                                                  0.021654
BAC                                                                 0.014034
BB.TO                                                               0.037817
BIIB                                                                0.021318
BLK                                                                 0.017138
BMY                              

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

In [18]:
print('Period:', prices.index[0].date(), 'to', prices.index[-1].date(), '(all values in CAD)')
print('Client A')
display(comparison(weights_a))
print('Client B')
display(comparison(weights_b))

Period: 2025-10-01 to 2026-09-30 (all values in CAD)
Client A


,Total return,Daily std,Annualized std
Chosen weights,0.498590,0.014383,0.228326
Equal weights,0.422811,0.012070,0.191599


Client B


,Total return,Daily std,Annualized std
Chosen weights,0.120981,0.006864,0.108963
Equal weights,0.102328,0.007094,0.112607


In [13]:
calm = standard_deviation.sort_values().index[:len(standard_deviation) * 6 // 10]
corr = returns[calm].corr()

chosen_b = [calm[0]]
sector_count = {info_stocks.loc[calm[0], 'Sector']: 1}
while len(chosen_b) < 18:
    best = None
    best_corr = 2
    for t in calm:
        sector = info_stocks.loc[t, 'Sector']
        if t in chosen_b or sector_count.get(sector, 0) >= 7:
            continue
        avg_corr = corr.loc[t, chosen_b].mean()
        if avg_corr < best_corr:
            best = t
            best_corr = avg_corr
    if best is None:
        break
    chosen_b.append(best)
    sector = info_stocks.loc[best, 'Sector']
    sector_count[sector] = sector_count.get(sector, 0) + 1
chosen_b

['RY.TO',
 'MO',
 'T.TO',
 'LMT',
 'AIG',
 'AXP',
 'ABBV',
 'AAPL',
 'UNP',
 'ABT',
 'C',
 'PM',
 'PFE',
 'TD.TO',
 'KO',
 'PEP',
 'BLK',
 'BMY']

In [17]:
raw_b = 1 / standard_deviation[chosen_b]
weights_b = apply_caps(raw_b, info_stocks.loc[chosen_b, 'Sector'])
Portfolio_Final_B, fees_b = build_portfolio(weights_b, 750000)
display(Portfolio_Final_B)

,Ticker,Price,Currency,Shares,Value,Weight
1,RY.TO,263.049988,CAD,247.781337,65178.877733,0.086906
2,MO,74.870003,USD,341.138916,36623.342499,0.048832
3,T.TO,15.860000,CAD,2651.459419,42052.145483,0.056070
4,LMT,565.500000,USD,42.323353,34318.756464,0.045759
5,AIG,79.680000,USD,353.411439,40378.370923,0.053839
6,AXP,301.700012,USD,83.831191,36266.014129,0.048355
7,ABBV,286.200012,USD,91.274830,37457.565352,0.049944
8,AAPL,342.559998,USD,80.427965,39505.957438,0.052675
9,UNP,287.290009,USD,104.857961,43195.727567,0.057595
10,ABT,103.000000,USD,244.707040,36141.198714,0.048189


## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

In [19]:
def validate(table, fees, budget, name):
    sectors = info_stocks.loc[table['Ticker'], 'Sector'].values
    sector_weights = table.groupby(sectors)['Weight'].sum()
    checks = {'Tickers are valid and eligible': set(table['Ticker']) <= set(filtered_tickers),
              'Between 10 and 20 stocks': 10 <= len(table) <= 20,
              'No stock above 15%': table['Weight'].max() <= 0.15 + 1e-9,
              'No sector above 40%': sector_weights.max() <= 0.40 + 1e-9,
              'Weights sum to 100%': abs(table['Weight'].sum() - 1) < 1e-9,
              'Purchases + fees within budget': table['Value'].sum() + fees <= budget}
    print(name)
    for check, passed in checks.items():
        print('  PASS' if passed else '  FAIL', check)
    assert all(checks.values())

validate(Portfolio_Final_A, fees_a, 1000000, 'Client A')
validate(Portfolio_Final_B, fees_b, 750000, 'Client B')


Client A
  PASS Tickers are valid and eligible
  PASS Between 10 and 20 stocks
  PASS No stock above 15%
  PASS No sector above 40%
  PASS Weights sum to 100%
  PASS Purchases + fees within budget
Client B
  PASS Tickers are valid and eligible
  PASS Between 10 and 20 stocks
  PASS No stock above 15%
  PASS No sector above 40%
  PASS Weights sum to 100%
  PASS Purchases + fees within budget


## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

In [20]:
print('Client A')
display(Portfolio_Final_A)
print('Client B')
display(Portfolio_Final_B)

reconciliation = pd.DataFrame({
    'Client A': [1000000, Portfolio_Final_A['Value'].sum(), fees_a, 1000000 - Portfolio_Final_A['Value'].sum() - fees_a],
    'Client B': [750000, Portfolio_Final_B['Value'].sum(), fees_b, 750000 - Portfolio_Final_B['Value'].sum() - fees_b]},
    index=['Budget', 'Purchases', 'Fees', 'Unspent'])
display(reconciliation)

Portfolio_Final_A[['Ticker', 'Shares']].to_csv('Stocks_Group_XX_A.csv', index=False)
Portfolio_Final_B[['Ticker', 'Shares']].to_csv('Stocks_Group_XX_B.csv', index=False)

Client A


,Ticker,Price,Currency,Shares,Value,Weight
1,CAT,510.450012,USD,204.934704,149998.733419,0.150000
2,BB.TO,11.100000,CAD,12772.020735,141769.435029,0.141771
3,TXN,263.750000,USD,306.525863,115925.360877,0.115926
4,QCOM,149.940002,USD,449.638517,96671.817698,0.096673
5,C,110.620003,USD,603.914773,95791.766584,0.095793
6,MRK,133.720001,USD,419.194364,80376.792506,0.080377
7,TD.TO,162.050003,CAD,431.846658,69980.752284,0.069981
8,AMZN,246.820007,USD,177.998257,62996.290221,0.062997
9,BIIB,249.589996,USD,136.179354,48736.833622,0.048737
10,RY.TO,263.049988,CAD,176.967541,46551.309610,0.046552


Client B


,Ticker,Price,Currency,Shares,Value,Weight
1,RY.TO,263.049988,CAD,247.781337,65178.877733,0.086906
2,MO,74.870003,USD,341.138916,36623.342499,0.048832
3,T.TO,15.860000,CAD,2651.459419,42052.145483,0.056070
4,LMT,565.500000,USD,42.323353,34318.756464,0.045759
5,AIG,79.680000,USD,353.411439,40378.370923,0.053839
6,AXP,301.700012,USD,83.831191,36266.014129,0.048355
7,ABBV,286.200012,USD,91.274830,37457.565352,0.049944
8,AAPL,342.559998,USD,80.427965,39505.957438,0.052675
9,UNP,287.290009,USD,104.857961,43195.727567,0.057595
10,ABT,103.000000,USD,244.707040,36141.198714,0.048189


,Client A,Client B
Budget,1000000.000000,750000.000000
Purchases,999991.556125,749990.501326
Fees,8.433875,9.488674
Unspent,0.010000,0.010000


## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.